# Lab | Agents and Tools
## A research assistant for the bank's product team

The bank from last week is planning its 2027 products, and the product team spends hours answering the same kind of questions: *How many customers are on Premium? What does the competition charge? How many complaints were about card delivery?* Some answers are in the bank's internal documents, some are on the web, and many need a quick calculation.

Your job is to build a **research assistant**: an agent that searches the internal documents, searches the web with Tavily, and does arithmetic with a calculator, and that **cites** where each fact came from. Then you **evaluate** it: does it choose the right tools, and what does each answer cost?

**The data** is `data/bank_documents.csv`: ten short internal documents of the bank (invented for this course), with the columns `id`, `title` and `text`.

**This lab practises** building and checking an agent. You will use, from today's **Agents and Tools** lesson: tools with `@tool`, `create_agent`, the message trace and its token counts, Tavily web search, a retrieval tool over Chroma, memory with a checkpointer, and step limits.

**By the end, you can:**

1. Implement an agent with three tools (a function of your own, a retrieval tool and web search) and read its trace.
2. Evaluate the agent's tool choices and cost on a test set, and explain its failures.

Each challenge ends with a **checkpoint** that tells you what you should see. Agents make their own choices, so your traces will not match the checkpoints exactly. **The whole lab costs about one US cent and fewer than 20 Tavily searches**, well within the free plan.

## Setup

You need `pip install openai python-dotenv langchain langchain-openai langchain-tavily sentence-transformers chromadb tiktoken`. Create a `.env` file in the `your-code` folder with your two keys, as in the lesson:

```
OPENAI_API_KEY=sk-...
TAVILY_API_KEY=tvly-...
```

The repository's `.gitignore` already lists `.env`: **never write a key in the notebook**, and check with `git status` before you commit that `.env` is not listed.

The cell below loads the libraries and the documents, and gives you two helpers. `run(agent, question)` runs an agent and returns a dictionary with the answer, the tools it called, the number of model calls and the tokens. `cost(input_tokens, output_tokens)` turns tokens into US dollars.

In [ ]:
import os
import logging
import warnings

os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)

import pandas as pd
import tiktoken
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_tavily import TavilySearch
from langgraph.checkpoint.memory import InMemorySaver
from sentence_transformers import SentenceTransformer
import chromadb
from transformers.utils import logging as transformers_logging

transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()
warnings.filterwarnings("ignore", message=".*torch.jit.script.*")
pd.set_option("display.max_colwidth", 100)

load_dotenv()
print("OpenAI key found:", "OPENAI_API_KEY" in os.environ, "| Tavily key found:", "TAVILY_API_KEY" in os.environ)
MODEL = "openai:gpt-5.4-nano"
PRICES = (0.20, 1.25)                        # US dollars per million input / output tokens (October 2026)

def cost(input_tokens, output_tokens):
    return (input_tokens * PRICES[0] + output_tokens * PRICES[1]) / 1_000_000

def run(agent, question, config=None):
    """Run an agent on one question and summarise what it did."""
    result = agent.invoke({"messages": [{"role": "user", "content": question}]}, config or {"recursion_limit": 12})
    ai_messages = [m for m in result["messages"] if type(m).__name__ == "AIMessage"]
    tokens_in = sum(m.usage_metadata["input_tokens"] for m in ai_messages)
    tokens_out = sum(m.usage_metadata["output_tokens"] for m in ai_messages)
    return {"answer": result["messages"][-1].content,
            "tools": [call["name"] for m in ai_messages for call in m.tool_calls],
            "model_calls": len(ai_messages), "input_tokens": tokens_in, "output_tokens": tokens_out,
            "cost": cost(tokens_in, tokens_out)}

documents = pd.read_csv("../data/bank_documents.csv")
documents

**Checkpoint:** both keys found, and a table of 10 documents, from `premium-1` to `ai-1`.

## Challenge 1: your first tool

1. Write a tool `calculator(expression: str) -> str` with the `@tool` decorator. Its docstring must tell the model what it does. For safety, it must **refuse** any expression that contains a character other than digits, spaces and `+ - * / ( ) .` (return an error message instead), and otherwise return the result of `eval(expression)`.
2. Test it directly with `calculator.invoke(...)` on `"182000 * 0.14"` and on `"__import__('os').getcwd()"`.
3. Create an agent with `create_agent`, the model `MODEL`, the calculator as its only tool and a short system prompt. Use `run` to ask it *What is 14% of 182,000, and what is that times 9.99?* and print the result dictionary.

In [ ]:
# 1. The calculator tool


# 2. Test it directly


# 3. A first agent

**Checkpoint:** `25480.0`, then the error message for the second expression: the tool refused to run Python code. The agent calls the calculator twice (14% of 182,000, then that times 9.99) and answers **25,480** and **254,545.2**, with three model calls, about 600 input tokens and a cost of about $0.0002.

## Challenge 2: a tool for the bank's documents

1. Load the MiniLM embedding model (`sentence-transformers/all-MiniLM-L6-v2`), embed `title. text` for every document (normalised), and store them in a Chroma collection with cosine distance and the document `id`s as ids.
2. Write a tool `search_documents(query: str) -> str` that returns the **two** closest documents, each on one line as `[id] title: text`. Its docstring must say what is in the documents, so that the agent knows when to use it.
3. Test it directly with the query *how many customers do we have*.

In [ ]:
# 1. Index the documents


# 2. The retrieval tool


# 3. Test it

**Checkpoint:** two lines, `[customers-1]` first (182,000 customers, 14% on Premium) and then the next closest document, `[complaints-1]` in the reference run.

## Challenge 3: add the web, and keep the results short

1. Create a Tavily search tool with `TavilySearch(max_results=3)`. Call it once directly with the query *Revolut Premium plan price per month Spain*, and count the **tokens** of the result with `tiktoken` (use `tiktoken.get_encoding("o200k_base")` and `len(encoding.encode(str(result)))`).
2. Long tool results are expensive, and every one is sent back to the model at each later step. Write a tool `web_search(query: str) -> str` that calls the Tavily tool, keeps only the `title`, `url` and `content` of each result, and **trims** the whole text to at most 500 tokens.
3. Create the research agent with `calculator`, `search_documents` and `web_search`. Its system prompt must say when to use each tool, ask it to cite the document id or the URL of every fact, and to answer in at most four sentences.
4. Use `run` on these three questions, and print the answer, the tools and the cost of each:
   - *How much monthly revenue does the Premium plan bring?*
   - *How does our Premium plan compare with Revolut's Premium plan in price?*
   - *Name two European fintech startups that raised money for AI products in 2026, with a source for each.*

In [ ]:
# 1. Tavily, and the size of one result


# 2. A trimmed web search tool


# 3. The research agent


# 4. Three questions

**Checkpoint:** about 900 tokens for one raw Tavily result, and 500 after trimming.

- **Revenue:** `search_documents` and `calculator`, about **254,545 euros per month**, citing `premium-1` and `customers-1`.
- **Revolut:** the documents and the web, sometimes the calculator too. Read the answer carefully: in the reference runs, the price came from another country (once in US dollars, once in UK pounds, from a comparison site) instead of Spain, and the agent said so.
- **Startups:** two or three web searches. In one reference run, the agent named two companies that are **not European**; in another, it said it could not find verifiable examples. Both are typical.

Each answer costs less than a tenth of a cent.

**Question:** why does trimming the search results matter, both for the cost and for the model? What could you lose by trimming too much?

*Your answer here.*

## Challenge 4: evaluate the tool choices

The product team wrote eight questions and, for each, the set of tools a good answer needs.

1. Run `research_agent` on every question in `TEST_SET` with `run`, and build a DataFrame with the question, the expected tools, the tools called (as a set), whether they are **exactly** the expected set (`right_tools`), the number of model calls, the tokens and the cost.
2. Print the share of questions with the right tools, the total cost, and the average cost of the questions that used `web_search` against those that did not.
3. Read the answers of the questions with the wrong tools.

In [ ]:
TEST_SET = [
    ("What does the Premium plan include?", {"search_documents"}),
    ("How many customers are on the Premium plan?", {"search_documents", "calculator"}),
    ("How many complaints were about card delivery last quarter?", {"search_documents", "calculator"}),
    ("What is 17.5% of 2,400?", {"calculator"}),
    ("What is the monthly price of Revolut's Premium plan in Spain?", {"web_search"}),
    ("What did the European Central Bank decide about interest rates at its latest meeting?", {"web_search"}),
    ("When will our app get an AI assistant, and what must it tell users under the EU AI Act?", {"search_documents", "web_search"}),
    ("Do Premium customers pay a fee on card payments abroad, and how much would a standard customer pay on 300 euros?", {"search_documents", "calculator"}),
]

# 1. Run the test set


# 2. Right tools and cost


# 3. The wrong ones

**Checkpoint:** 7 of the 8 questions (88%) with exactly the expected tools, for a total of about a third of a cent; in the reference run the web questions cost a little more on average (about $0.0004 against $0.0003, thanks to the trimming). The miss was the card-delivery question: the agent found `complaints-1` and computed 38% of 1,240 = 471 **by itself**, without the calculator.

**Question:** look at the questions where the tools were not exactly the expected ones. Was the answer wrong, or was the test too strict? What would you change: the tool descriptions, the system prompt, or the test?

*Your answer here.*

## Challenge 5: memory and limits

1. Create `chat_agent`: the same tools and system prompt, plus `checkpointer=InMemorySaver()`.
2. Hold this conversation in one thread (`{"configurable": {"thread_id": "product-team"}, "recursion_limit": 12}`), printing each answer:
   - *How many complaints did we get last quarter?*
   - *What share of them were about blocked cards, and how many is that?*
   - *Remind me: what was the total you started from?*
3. Ask the research agent *Compare every one of our documents with what our three main competitors offer, one by one.* with `{"recursion_limit": 4}`, and catch the `GraphRecursionError` (import it from `langgraph.errors`). What should a real application show the user when this happens?

In [ ]:
# 1. An agent with memory


# 2. A conversation in one thread


# 3. A step limit

**Checkpoint:** 1,240 complaints; then 22%, about 273 (272.8); then 1,240 again, which proves the memory works. The broad question prints `Stopped by the step limit`. A real application should say that the question was too broad and suggest a narrower one (one product, one competitor), not show a half-finished answer, and it should log the case.

## What did it cost?

Add up the cost of the evaluation in Challenge 4 and compare it with the price of the team's time.

**Question:** the test set cost a fraction of a cent per question. If the product team asks 200 questions a day, about how much would the assistant cost per month? Which part of the cost would you watch, and what is the cost you cannot see in the token count?

*Your answer here.*

## Extra: give the assistant a voice

From the **Multimodal AI** lesson:

1. Generate a spoken question with `gpt-4o-mini-tts`, for example *How many of our customers are on the Premium plan?*, and save it as `question.mp3`.
2. Transcribe it with `gpt-4o-mini-transcribe`, send the transcript to `research_agent`, and turn the answer into speech.
3. Listen to the answer. Is a four-sentence answer with citations a good spoken answer? Change the system prompt for a voice assistant and try again.

In [ ]:
# Your code here

**Conclusions:** *Your answer here.*